# Sionna 0.19 — 915.95 MHz Simulation (Nottingham)

**Frequency:** 915.95 MHz · **RX:** 1200 sequential from CSV · **Terrain:** flat · **Materials:** metal roof, brick walls, wet ground

**CSV:** nottingham915.csv  ·  TX EIRP=55.1 dBm  ·  RX system gain=−7.8 dB

## Cell 0 — Imports

In [ ]:
import os, sys, json, csv, time, warnings, glob, re
import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.constants import speed_of_light as C
from pyproj import Transformer
import math
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

FORCE_CPU_RT = False
if not FORCE_CPU_RT:
    import mitsuba as mi
    mi.set_variant('cuda_ad_rgb')
else:
    import mitsuba as mi
    mi.set_variant('llvm_ad_rgb')
import drjit as dr

import sionna
import tensorflow as tf
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver
from sionna.rt.antenna import iso_pattern, dipole_pattern
print(f'Sionna {sionna.__version__}  TF {tf.__version__}  Mitsuba {mi.__version__}')

## Cell 1 — Configuration

In [ ]:
# ── City / scene ─────────────────────────────────────────────────────────────
CITY_NAME    = 'Nottingham'
UTM_EPSG     = 32630          # UTM zone 30N (UK)

# ── Scene bbox (WGS84) ────────────────────────────────────────────────────────
# Tight bbox: TX + first 1200 RX + 900m margin (10.1×7.3 km = 74 km²)
SCENE_WEST   = -1.270568
SCENE_EAST   = -1.118832
SCENE_SOUTH  =  52.939492
SCENE_NORTH  =  53.005008

# ── TX parameters (from nottingham915.csv header) ─────────────────────────────
# Amp power=50.3 dBm, cable loss=1.3 dB → conducted=49.0 dBm
# Antenna gain=1.3 dBi → EIRP=55.1 dBm  (matches CSV header)
TX_LON           = -1.2559
TX_LAT           =  52.9863
TX_AGL_M         = 17.0           # m  (CSV: Tx antenna height = 17 m)
TX_CONDUCTED_DBM = 49.0           # dBm (50.3 amp − 1.3 cable loss)
TX_ANTENNA_GAIN_DBI = 1.3         # dBi
EIRP_DBM         = TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI  # = 50.3 dBm ≈ 55.1 per CSV header

# ── RX parameters (from nottingham915.csv header) ─────────────────────────────
# RX antenna gain=-1 dBi, cable loss=0.2 dB, splitter=6.1 dB,
# LNA=0 dB, BPF=0.5 dB → system gain = -1 - 0.2 - 6.1 + 0 - 0.5 = -7.8 dB
RX_AGL_M          = 1.5           # m  (CSV: Rx antenna height = 1.5 m)
RX_EXTRA_GAIN_DB  = -7.8          # dB (system gain from CSV)
SITE_CORRECTION_DB = 0.0          # dB (calibration offset — adjust after seeing bias)
NOISE_FLOOR_DBM   = -124.0        # dBm (CSV: System noise floor)

# ── RX selection ──────────────────────────────────────────────────────────────
NUM_RX       = 1200              # first 1200 sequential rows from CSV

# ── Frequency ─────────────────────────────────────────────────────────────────
FREQUENCY_HZ = 915.95e6          # Hz  (CSV: Frequency = 915.95 MHz)

# ── Terrain ───────────────────────────────────────────────────────────────────
FLAT_TERRAIN = True              # flat z=0 plane

# ── Ray tracing ───────────────────────────────────────────────────────────────
MAX_DEPTH        = 6             # bounces
NUM_SAMPLES_PS   = 10_000_000    # rays per batch
SCAT_KEEP_PROB   = 0.001         # scatter fraction (energy corrected)
BATCH_SIZE       = 5             # RX per compute_paths() call

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR        = '/home/georgeskai/Documents/FYP2026/nottingham900'
SCENE_DIR       = os.path.join(BASE_DIR, 'scene')
SCENE_XML       = os.path.join(SCENE_DIR, 'scene.xml')
OUT_DIR         = os.path.join(BASE_DIR, 'results')
os.makedirs(OUT_DIR, exist_ok=True)

# Source measurement CSV (nottingham915.csv)
OFCOM_RAW_CSV   = '/home/georgeskai/Documents/FYP2026/nottingham900/nottingham915.csv'

# Output files (same naming convention as main notebook)
RX_CSV          = os.path.join(SCENE_DIR, 'receiver_locations.csv')
MEASUREMENT_CSV = os.path.join(SCENE_DIR, 'measurements_with_pathloss.csv')

print(f'Frequency        : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'TX               : lon={TX_LON}  lat={TX_LAT}  AGL={TX_AGL_M}m')
print(f'TX conducted     : {TX_CONDUCTED_DBM} dBm   EIRP={EIRP_DBM:.1f} dBm')
print(f'RX system gain   : {RX_EXTRA_GAIN_DB} dB')
print(f'Noise floor      : {NOISE_FLOOR_DBM} dBm')
print(f'NUM_RX           : {NUM_RX}')
print(f'BASE_DIR         : {BASE_DIR}')
print(f'SCENE_XML        : {SCENE_XML}')
print(f'OFCOM_RAW_CSV    : {OFCOM_RAW_CSV}')


## Cell 2 — Coordinate Utilities (GPS → UTM → Local, NO BNG)

In [ ]:
# ── Coordinate transformers (pyproj, WGS84 ↔ UTM 30N) ──────────────────────
# Best practice: always_xy=True enforces (lon, lat) / (easting, northing) order
# regardless of the CRS axis convention — prevents silent axis swaps.
gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

# Scene centre in UTM (origin of local coordinate system)
center_lon = (SCENE_WEST + SCENE_EAST)  / 2
center_lat = (SCENE_SOUTH + SCENE_NORTH) / 2
utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)

def gps_to_local(lon, lat, height=0.0):
    """WGS84 (lon, lat) → scene-local (x, y, z) in metres.
    Origin = scene bbox centre. X = east, Y = north, Z = up.
    """
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    """Scene-local (x, y) → WGS84 (lon, lat)."""
    lon, lat = utm_to_gps.transform(x + utm_center_x, y + utm_center_y)
    return float(lon), float(lat)

print(f'Scene centre  : lon={center_lon:.5f}  lat={center_lat:.5f}')
print(f'UTM centre    : ({utm_center_x:.1f}, {utm_center_y:.1f})')
print(f'Test gps_to_local(TX): {gps_to_local(TX_LON, TX_LAT)[:2]}')

## Cell 3 — Load Scene

In [ ]:
print(f'Loading scene: {SCENE_XML}')
scene = load_scene(SCENE_XML)
scene.frequency       = FREQUENCY_HZ
scene.synthetic_array = False

# Antenna arrays — isotropic for simplicity at 900 MHz
_iso = iso_pattern
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                              vertical_spacing=0.5, horizontal_spacing=0.5,
                              pattern=_iso, polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                              vertical_spacing=0.5, horizontal_spacing=0.5,
                              pattern=_iso, polarization='V')
print(f'Scene loaded  : {len(scene.objects)} objects')
print(f'Frequency     : {FREQUENCY_HZ/1e6:.0f} MHz')

## Cell 4 — Materials (Metal Roof, Brick Walls, Wet Ground)

In [ ]:
# ── ITU-R P.2040-2 at 900 MHz ────────────────────────────────────────────────
# eps_r and sigma computed at FREQUENCY_HZ
_f_ghz = FREQUENCY_HZ / 1e9   # 0.9 GHz

def _itu(a_eps, b_eps, c_sig, d_sig):
    eps   = a_eps * (_f_ghz ** b_eps)
    sigma = c_sig * (_f_ghz ** d_sig)
    return float(eps), float(sigma)

_mats = {
    # name              a_eps  b_eps   c_sig   d_sig     S_scat  note
    'itu_brick'      : (*_itu(3.75, 0, 0.038,  0    ), 0.20),   # walls
    'itu_metal'      : (*_itu(1.0,  0, 1e7,    0    ), 0.05),   # roofs (specular)
    'itu_wet_ground' : (*_itu(30.0, 0, 0.15,   0    ), 0.10),   # floor/terrain
    'itu_concrete'   : (*_itu(5.31, 0, 0.0326, 0.8095), 0.15), # generic
    'itu_glass'      : (*_itu(6.27, 0, 0.0043, 1.1925), 0.05), # windows
    'itu_vegetation' : (*_itu(1.7,  0, 0.050,  0.60 ), 0.50),   # trees
    'itu_asphalt'    : (*_itu(3.18, 0, 0.058,  0    ), 0.10),   # roads
}

from sionna.rt.scattering_pattern import LambertianPattern as _LP
_lambertian = _LP()

for _name, (_eps, _sig, _s) in _mats.items():
    _mat = RadioMaterial(_name,
                         relative_permittivity=_eps,
                         conductivity=_sig,
                         scattering_coefficient=_s,
                         xpd_coefficient=0.0,
                         scattering_pattern=_lambertian)
    scene.add(_mat)
    print(f'  {_name:<22} eps={_eps:.2f}  sigma={_sig:.4f}  S={_s:.2f}')

# Assign materials to scene objects by name pattern
# Roof → metal, Wall → brick, Ground/terrain → wet_ground
for _obj_name, _obj in scene.objects.items():
    _n = _obj_name.lower()
    if 'roof' in _n:
        _obj.radio_material = scene.get(_name='itu_metal')
    elif 'wall' in _n or 'bld' in _n or 'building' in _n:
        _obj.radio_material = scene.get(_name='itu_brick')
    elif 'terrain' in _n or 'ground' in _n or 'floor' in _n:
        _obj.radio_material = scene.get(_name='itu_wet_ground')

print('\nMaterial assignment done.')
print('  Roof  → itu_metal  (perfect conductor, specular reflection)')
print('  Walls → itu_brick  (eps=3.75, sigma=0.038 @ 900 MHz)')
print('  Floor → itu_wet_ground  (eps=30.0, sigma=0.15 @ 900 MHz)')
print()
print('NOTE: For sloped/pitched roof geometry, rebuild scene with')
print('      roof_type=pitched in the scene builder (Cell 4 _extrude_roof).')

## Cell 5 — Place TX (Flat Terrain, z=0)

In [ ]:
# Remove previous TX
for _n in list(scene.transmitters.keys()):
    scene.remove(_n)

tx_x, tx_y, _ = gps_to_local(TX_LON, TX_LAT)
tx_z = TX_AGL_M   # flat terrain: ground = 0

tx = Transmitter(name='tx0',
                 position=[tx_x, tx_y, tx_z],
                 orientation=[0.0, 0.0, 0.0])
scene.add(tx)

print(f'TX placed:')
print(f'  GPS      : lon={TX_LON}  lat={TX_LAT}')
print(f'  Local    : ({tx_x:.1f}, {tx_y:.1f}, {tx_z:.1f}) m')
print(f'  AGL      : {TX_AGL_M} m  (flat terrain)')

## Cell 6 — RX Extraction (1200 Nearest, Sequential)

In [ ]:
import csv as _csv_mod, math

print('=' * 60)
print('CELL 6 — RX EXTRACTION (first 1200 sequential from CSV)')
print('=' * 60)

if not os.path.exists(OFCOM_RAW_CSV):
    raise FileNotFoundError(f'CSV not found: {OFCOM_RAW_CSV}')

# ── Detect header row (skip site metadata lines) ──────────────────────────────
_CSV_HEADER_ROW = 21   # nottingham915.csv: 21 metadata lines before column headers
_df_raw = pd.read_csv(OFCOM_RAW_CSV, skiprows=_CSV_HEADER_ROW, low_memory=False)
print(f'Columns: {list(_df_raw.columns)}')

# Column mapping for nottingham915.csv
_lat_col  = 'Rx Latitude (deg)'
_lon_col  = 'Rx Longitude (deg)'
_rssi_col = 'Local mean measurement (dBm)'
_date_col = 'Date (dd.mm.yyy)'
_time_col = 'Time (hh:mm:ss)'

# Verify columns exist
for _col in [_lat_col, _lon_col, _rssi_col]:
    if _col not in _df_raw.columns:
        raise KeyError(f'Column not found: {_col!r}. Available: {list(_df_raw.columns)}')

# Distance from TX (for reference only — we take first 1200 rows sequentially)
_dlon_m = 111000.0 * math.cos(math.radians(TX_LAT))
_dlat_m = 111000.0
_df_raw['_dist_km'] = (
    ((_df_raw[_lat_col] - TX_LAT) * _dlat_m)**2 +
    ((_df_raw[_lon_col] - TX_LON) * _dlon_m)**2
)**0.5 / 1000.0

# Take first NUM_RX rows (CSV is already ordered by measurement time near TX)
_sel = _df_raw.head(NUM_RX).copy()
print(f'Selected  : {len(_sel)} receivers (first {NUM_RX} rows)')
print(f'Dist range: {_sel["_dist_km"].min():.3f} – {_sel["_dist_km"].max():.3f} km')
print(f'RSSI range: {_sel[_rssi_col].min():.1f} – {_sel[_rssi_col].max():.1f} dBm')

# ── Write receiver_locations.csv (same format as main notebook) ───────────────
os.makedirs(os.path.dirname(RX_CSV), exist_ok=True)
with open(RX_CSV, 'w', newline='') as _f:
    _w = _csv_mod.writer(_f)
    _w.writerow(['name', 'lon', 'lat', 'height'])
    for _idx, _row in _sel.iterrows():
        _w.writerow([f'RX_{_idx:06d}',
                     f'{float(_row[_lon_col]):.6f}',
                     f'{float(_row[_lat_col]):.6f}',
                     RX_AGL_M])
print(f'Written : {RX_CSV}')

# ── Write measurements_with_pathloss.csv (same format as main notebook) ────────
with open(MEASUREMENT_CSV, 'w', newline='') as _f:
    _w = _csv_mod.writer(_f)
    _w.writerow(['name', 'lon', 'lat', 'local_measurement_dBm', 'path_loss_dB'])
    for _idx, _row in _sel.iterrows():
        _rssi = float(_row[_rssi_col]) if pd.notna(_row[_rssi_col]) else float('nan')
        _pl   = EIRP_DBM - _rssi + RX_EXTRA_GAIN_DB if not math.isnan(_rssi) else float('nan')
        _w.writerow([f'RX_{_idx:06d}',
                     f'{float(_row[_lon_col]):.6f}',
                     f'{float(_row[_lat_col]):.6f}',
                     f'{_rssi:.2f}',
                     f'{_pl:.2f}'])
print(f'Written : {MEASUREMENT_CSV}')


## Cell 7 — Place Receivers

In [ ]:
import time as _time

df_rx = pd.read_csv(RX_CSV)
for nm in list(scene.receivers.keys()):
    scene.remove(nm)

receivers = []
for _, row in df_rx.iterrows():
    lx, ly, _ = gps_to_local(float(row['lon']), float(row['lat']))
    lz = RX_AGL_M   # flat terrain
    rx = Receiver(name=row['name'], position=[lx, ly, lz])
    scene.add(rx)
    receivers.append(rx)

print(f'Placed {len(receivers)} receivers at z={RX_AGL_M}m (flat terrain)')

## Cell 8 — Path Solver (900 MHz, All Receivers)

In [ ]:
import gc, math

_safe = lambda v: float(v) if not hasattr(v,'numpy') else float(v.numpy())
tx_pos = np.array([_safe(tx.position[0]), _safe(tx.position[1]), _safe(tx.position[2])])

PS_CONFIG = dict(
    max_depth        = MAX_DEPTH,
    los              = True,
    reflection       = True,
    diffraction      = True,
    edge_diffraction = True,
    scattering       = True,
    scat_keep_prob   = SCAT_KEEP_PROB,
)

results = []
total = len(receivers)
t0 = time.time()
print(f'Path solver: {total} RX  |  {NUM_SAMPLES_PS//1_000_000}M samples  |  depth={MAX_DEPTH}  |  900 MHz')
print('-' * 60)

for b_start in range(0, total, BATCH_SIZE):
    batch = receivers[b_start:b_start+BATCH_SIZE]
    # Remove all, add batch
    for nm in list(scene.receivers.keys()): scene.remove(nm)
    for rx in batch: scene.add(rx)
    try:
        paths = scene.compute_paths(**PS_CONFIG, num_samples=NUM_SAMPLES_PS)
        a_raw = paths.a
        if isinstance(a_raw, tuple): a = a_raw[0].numpy() + 1j*a_raw[1].numpy()
        else: a = np.array(a_raw)
        # Scatter amplitude correction
        try:
            types = np.array(paths.types)
            scat_mask = (types == 3)
            if scat_mask.any():
                a[scat_mask] *= np.sqrt(1.0 / SCAT_KEEP_PROB)
        except Exception: pass
        # Per-RX RSSI
        for bi, rx in enumerate(batch):
            dist = float(np.linalg.norm(
                np.array([_safe(rx.position[0]),_safe(rx.position[1]),_safe(rx.position[2])]) - tx_pos))
            try:
                a_rx = a[:, bi, :, :, 0, 0] if a.ndim >= 5 else a
                pwr  = float(np.sum(np.abs(a_rx)**2))
            except Exception:
                pwr = 0.0
            if pwr > 0:
                rssi = TX_CONDUCTED_DBM + 10*math.log10(pwr) + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB
                n_paths = int(np.sum(np.abs(a_rx) > 0))
            else:
                rssi = float('nan'); n_paths = 0
            results.append({'name': rx.name, 'dist_m': dist,
                            'rssi_sim_dbm': rssi, 'n_paths': n_paths})
    except Exception as e:
        for rx in batch:
            results.append({'name': rx.name, 'dist_m': 0, 'rssi_sim_dbm': float('nan'), 'n_paths': 0})
        print(f'  [WARN] batch {b_start}: {e}')
    del paths; gc.collect()
    done = min(b_start+BATCH_SIZE, total)
    if done % 50 < BATCH_SIZE or done == total:
        print(f'  [{done:4d}/{total}]  {time.time()-t0:.0f}s elapsed')

# Re-add all receivers
for nm in list(scene.receivers.keys()): scene.remove(nm)
for rx in receivers: scene.add(rx)

df_sim = pd.DataFrame(results)
_out = os.path.join(OUT_DIR, 'rssi_sim_900mhz.csv')
df_sim.to_csv(_out, index=False)
print(f'\nSaved: {_out}')
print(f'Valid RSSI: {df_sim["rssi_sim_dbm"].notna().sum()} / {len(df_sim)}')
print(f'RSSI range: {df_sim["rssi_sim_dbm"].min():.1f} – {df_sim["rssi_sim_dbm"].max():.1f} dBm')

## Cell 9 — Compare vs Measurements

In [ ]:
if not MEASUREMENT_CSV or not os.path.exists(MEASUREMENT_CSV):
    print('Set MEASUREMENT_CSV in Cell 1 to compare vs measurements.')
else:
    df_meas = pd.read_csv(MEASUREMENT_CSV)
    df_merge = df_sim.merge(df_meas[['name','local_measurement_dBm']], on='name', how='inner')
    df_merge = df_merge.dropna(subset=['rssi_sim_dbm','local_measurement_dBm'])
    df_merge['err'] = df_merge['rssi_sim_dbm'] - df_merge['local_measurement_dBm']

    bias = df_merge['err'].mean()
    rmse = (df_merge['err']**2).mean()**0.5
    print(f'Receivers compared : {len(df_merge)}')
    print(f'Bias (sim-meas)    : {bias:+.2f} dB')
    print(f'RMSE               : {rmse:.2f} dB')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].scatter(df_merge['dist_m']/1000, df_merge['err'], s=8, alpha=0.5)
    axes[0].axhline(0, color='red', lw=1)
    axes[0].set_xlabel('Distance (km)'); axes[0].set_ylabel('Error (dB)')
    axes[0].set_title(f'Prediction error vs distance  (bias={bias:+.1f} dB, RMSE={rmse:.1f} dB)')

    axes[1].scatter(df_merge['local_measurement_dBm'], df_merge['rssi_sim_dbm'], s=8, alpha=0.5)
    _lo = min(df_merge['local_measurement_dBm'].min(), df_merge['rssi_sim_dbm'].min())
    _hi = max(df_merge['local_measurement_dBm'].max(), df_merge['rssi_sim_dbm'].max())
    axes[1].plot([_lo,_hi],[_lo,_hi],'r--',lw=1)
    axes[1].set_xlabel('Measured RSSI (dBm)'); axes[1].set_ylabel('Simulated RSSI (dBm)')
    axes[1].set_title('Sim vs Measured  (900 MHz)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'rssi_compare_900mhz.png'), dpi=120)
    plt.show()